In [ ]:
import numpy as np
import seaborn as sb
import pandas
import sys
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.datasets import load_boston
from sklearn.linear_model import LinearRegression
%matplotlib notebook

# Simple two-layer, linear perceptron

We are going to implement a simple, linear perceptron with a linear activation function and train it using backpropagation.

Weights are initialized with random numbers of mean 0 and then update iteratively using the error between network output and wanted labels. 

Initially, we are going to test very simple input-output relationships with the input consisting of a 2-dimensional vector with either 0 or 1, and the output consisting of a 1-dimensional target-value also with either 0 or 1.

In [ ]:
# a linear activation function
def step(x):
    return ....

# implements a simple two-layer network
def twoLayer(X,y,plotting=True):
    # seed random numbers to get repeatable results
    np.random.seed(1)

    # initialize weights randomly (mean 0)
    # the number of weights is determined by the number
    # of columns in both the input data X and the output
    # data y!
    syn0 = 2*np.random.random((X.shape[1],y.shape[1])) - 1

    # maximum iteration
    maxIter = 100

    # store errors
    l1ErrorArray = np.zeros(maxIter)

    # do some iterations
    for it in np.arange(maxIter):

        # forward propagation: we put in our pattern
        # as layer "0" and then push it through the
        # activation function to get the output of
        # the layer
        l0 = X
        l1 = ....

        # evaluate the error of the layer
        l1Error = ....

        # evaluate the summed squared error
        l1ErrorArray[it] = ....

        # print out the summed squared error sometimes
        if (it%10==0):
            print("Iteration {:d}: error = {:f}\r".format(it,l1ErrorArray[it]))

        # the error determines the amount we need
        # to move 
        l1Delta = l1Error

        # the weight update is the dot product between
        # the pattern input and the correction amount 
        # times the learning rate
        lr = 0.01
        syn0 += ....

    print("\noutput after training is:\n",l1)
    print("\nweights after training are:\n",syn0)
    if (plotting):
        fig,ax = plt.subplots(figsize=(8,6))
        plt.plot(l1ErrorArray)
        plt.xlabel('Iteration')
        plt.ylabel('Summed Squared Error')
        plt.grid()
    return(syn0,l1ErrorArray)
    
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0],
                [1,1],
                [1,0],
                [0,1] ])
    
# this is the target state we want to have, 
# our input is three numbers, our output is
# one number
# note, that each row here corresponds to the
# row in X - so we are trying to basically
# learn two class labels from data here
y = np.array([[0],
              [1],
              [1],
              [0]])

# let's call our function and do the training
(weights,errors)=twoLayer(X,y)

That worked nicely. Let's try another set of simple target values for y. 

In [ ]:
y = np.array([[1],
              [0],
              [1],
              [0]])

# let's call our function and do the training
(weights,errors)=twoLayer(X,y)

Wait, that did not work at all - in fact, we have a large error, even though the shape of the convergence curve looks exactly the same.

In fact, we can see this since the first example that worked has resulting weights in which the first neuron gets a weight of $w_1\approx 1$, whereas the second neuron gets a weight of $w_2\approx 0$, so that the first column of the data (which goes to the first neuron) can simply be reproduced regardless of the input from the second neuron!!

But we forgot something! Right now, what we can do is to change the linear weights in our perceptron. But we have no way to shift the whole curve!! This is what the "bias" neuron can do.

To see why this is important, imagine we are trying to fit a line by only having weights:

$ y = w*x$

This can of course only change the slope of the line. But we also need to shift the line, so we need an intercept or bias:

$ y = w*x + b$


Let's add this neuron to our two-layer network:

In [ ]:
# implements a simple two-layer network
def twoLayerBias(X,y,plotting=True):
    # seed random numbers to get repeatable results
    np.random.seed(2)

    # let's add ones to the data to model the bias
    X=np.hstack((np.ones((X.shape[0],1)),X))
    
    # initialize weights randomly (mean 0)
    # the number of weights is determined by the number
    # of columns in both the input data X and the output
    # data y!
    syn0 = 2*np.random.random((X.shape[1],y.shape[1])) - 1
    print(syn0)
    # maximum iteration
    maxIter = 100

    # store errors
    l1ErrorArray = np.zeros(maxIter)

    # do some iterations
    for it in np.arange(maxIter):

        # forward propagation: we put in our pattern
        # as layer "0" and then push it through the
        # activation function to get the output of
        # the layer
        l0 = X
        l1 = step(np.dot(l0,syn0))
        
        # evaluate the error of the layer
        l1Error = y - l1
        
        # evaluate the summed squared error
        l1ErrorArray[it] = np.sum(l1Error*l1Error)

        # print out the summed squared error sometimes
        if (it%10==0):
            print("Iteration {:d}: error = {:f}\r".format(it,l1ErrorArray[it]))

        # the error determines the amount we need
        # to move along the derivative
        l1Delta = l1Error

        # the weight update is the dot product between
        # the pattern input and the correction amount 
        # times the learning rate
        lr = 0.1
        syn0 += lr*np.dot(l0.T,l1Delta)

    print("output after training is:\n",l1)
    print("\nweights after training are:\n",syn0)
    if (plotting):
        fig,ax = plt.subplots(figsize=(8,6))
        plt.plot(l1ErrorArray)
        plt.xlabel('Iteration')
        plt.ylabel('Summed Squared Error')
        plt.grid()
    return(syn0,l1ErrorArray)

In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0],
                [1,1],
                [1,0],
                [0,1] ])
    
y = np.array([[1],
              [0],
              [1],
              [0]])

# let's call our function and do the training
(weights,errors)=twoLayerBias(X,y)

Right, so now the curve can shift "up and down", because we can change the bias as well...

The y-labels we wanted the network to learn are actually logical combinations like ```AND``` or ```OR```, so let's make that explicit.

In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0],
                [1,1],
                [1,0],
                [0,1] ])
    
y = np.logical_and(X[:,0],X[:,1]).reshape(-1,1).astype(int)
print(y)
# let's call our function and do the training
(weights,errors)=twoLayerBias(X,y)

# Simple two-layer, non-linear perceptron

We are going to implement a simple, perceptron with a non-linear sigmoid activation function and train it using backpropagation.

Weights are initialized with random numbers of mean 0 and then update iteratively using the error between network output and wanted labels. 

Initially, we are going to test very simple input-output relationships with the input consisting of a 3-dimensional vector with either 0 or 1, and the output consisting of a 1-dimensional target-value also with either 0 or 1.

In [ ]:
def sigmoid(x):
    return....

def dsigmoid(x):
    return....

def twoLayerBias(X,y,lr=0.5,plotting=True):
    # seed random numbers to get repeatable results
    np.random.seed(1)
    
    # let's add ones to the data to model the bias
    X=np.hstack((np.ones((X.shape[0],1)),X))
    # initialize weights randomly (mean 0)
    # the number of weights is determined by the number
    # of columns in both the input data X (but remember we
    # have the bias). We still connect of course to the 
    # output dimensions determined by the data y 
    syn0 = 2*np.random.random((X.shape[1],y.shape[1])) - 1

    # maximum iteration
    maxIter = 5000

    # store errors
    l1ErrorArray = np.zeros(maxIter)

    # do some iterations
    for it in np.arange(maxIter):

        # forward propagation: we put in our pattern
        # as layer "0" and then push it through the
        # activation function to get the output of
        # the layer
        l0 = X
        l1 = sigmoid(np.dot(l0,syn0))

        # evaluate the error of the layer
        l1Error = y - l1

        # evaluate the summed squared error
        l1ErrorArray[it] = np.sum(l1Error*l1Error)

        # print out the summed squared error sometimes
        if (it%100==0):
            print("Iteration {:d}: error = {:f}".format(it,l1ErrorArray[it]))

        # the error determines the amount we need
        # to move along the derivative
        l1Delta = ....

        # the weight update is the dot product between
        # the pattern input and the correction amount times
        # the learning rate
        syn0 += lr*np.dot(l0.T,l1Delta)

    print("output after training is:\n",l1)
    print("\nweights after training are:\n",syn0)
    if (plotting):
        fig,ax = plt.subplots(figsize=(8,6))
        plt.plot(l1ErrorArray)
        plt.xlabel('Iteration')
        plt.ylabel('Summed Squared Error')
        plt.grid()
    return(syn0,l1ErrorArray)


In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0,1],
                [1,1,1],
                [1,0,1],
                [0,1,1],
                [1,0,0],
                [1,1,0],
                [0,1,0],
                [0,0,0]])
    
# this is the target state we want to have, 
# our input is three numbers, our output is
# one number
# here is a simple, logical combination:
y = np.logical_or(X[:,0],np.logical_and(X[:,1],X[:,2]))
print(y)

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y.reshape(-1,1))
print(weights)

Cool, that works.

So let's try something very different. Let's go back to a very standard IRIS dataset, which contains measurements of different iris-flowers and their corresponding labels that group the flowers into different types.

We want to categorize this data using our two-layer network:

In [ ]:
iris = load_iris()
# these are the inputs to our neural network
# each row is one training example
X = iris.data[:100]
    
# this is the target state we want to have
y = iris.target[:100]

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y.reshape(-1,1))

Ugh. That did not go so well. Somehow, the network did not learn. Instead the error goes pretty much immediately to high values. What went wrong?

The problem is in the input values or in the learning rate. Remembert that we are doing gradient descent, so when we use the gradient for updating the weights, the gradient can become too big and the network "overshoots" and cannot find the correct minimum. 

There are two solutions:

* normalize the data: `X = (X-X.mean(axis=0))/X.std(axis=0)`
* change the learning rate

Let's try the first one:

In [ ]:
iris = load_iris()
# these are the inputs to our neural network
# each row is one training example
X = iris.data[:100]
    
# normalize the data
X = (X-X.mean(axis=0))/X.std(axis=0)

# this is the target state we want to have
y = iris.target[:100]

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y.reshape(-1,1))

That seemed to have done the trick.

Let's try the second one without normalizing the data:

In [ ]:
iris = load_iris()
# these are the inputs to our neural network
# each row is one training example
X = iris.data[:100]
    
# this is the target state we want to have
y = iris.target[:100]

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y.reshape(-1,1),.01)

And that works as well!

**Remember: gradient descent needs proper step-sizes!**

## Breaking the neural network

Let's return to our simple, 0/1 example and try a different training/testing combination:

In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0,1],
                [1,1,1],
                [1,0,1],
                [0,1,1] ])
    
# this is the target state we want to have, 
# our input is three numbers, our output is
# one number
# note, that each row here corresponds to the
# row in X - so we are trying to basically
# learn two class labels from data here
y = np.array([[1],
              [1],
              [0],
              [0]])

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y,.1)

Ouch. So that does not work. We have left the realm of simple correlations and the output space is a weird, highly non-linear combination of the inputs.

But this is a sigmoid-network, so can our network even learn non-linear things?

In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0,1],
                [1,1,1],
                [1,0,1],
                [0,1,1] ])
    
# this is the target state we want to have, 
# our input is three numbers, our output is
# one number
# here is an explictly non-linear function:
y = (X[:,0]+(X[:,1]-X[:,2])/(X[:,1]+0.4))
print(y)

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y.reshape(-1,1))
print(weights)

Apparently not. But wait!

If we look at the "y" target values, we can see that they are well outside the 0,1 range of the network inputs. Since the activation function itself is normalized between 0 and 1, we of course should normalize our output to be between 0 and 1:

In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0,1],
                [1,1,1],
                [1,0,1],
                [0,1,1] ])
    
# this is the target state we want to have, 
# our input is three numbers, our output is
# one number
# here is an explictly non-linear function:
y = (X[:,0]+(X[:,1]-X[:,2])/(X[:,1]+0.4))
print(y)
y = (y-y.min())/(y.max()-y.min())
print(y)

# let's call our function and do the training:
(weights,errors)=twoLayerBias(X,y.reshape(-1,1))
print(weights)

But, still, the example from above cannot apparently be learned. This is because even though we have non-linear activation, the target vectors are far outside the "span" of the input examples.

The solution? 

More layers!

## Three-layer network

Let's add a "hidden" layer to our network. We also add a bias node to the input layer of the network in order to be able to shift the outputs around.

In [ ]:
def threeLayerBias(X,y,numHidden=5,lr=0.02,plotting=True):

    # seed random numbers to get repeatable results
    np.random.seed(1)

    # let's add ones to the data to model the bias
    X=np.hstack((np.ones((X.shape[0],1)),X))
    
    # initialize weights randomly (mean 0)
    syn0 = 2*np.random.random((X.shape[1],numHidden)) - 1
    syn1 = 2*np.random.random((numHidden,y.shape[1])) - 1

    # maximum iteration
    maxIter = 40000

    # store errors
    l2ErrorArray = np.zeros(maxIter)

    # do some iterations
    for it in np.arange(maxIter):

        # forward propagation: we put in our pattern
        # as layer "0" and then push it through the
        # activation function to get the output of
        # the layer
        l0 = X
        l1 = ....
        l2 = ....

        # evaluate the error of the layer
        l2Error = ....

        l2ErrorArray[it] = np.sum(l2Error*l2Error)

        if (it%1000==0):
            sys.stdout.write("Iteration {:d}: error = {:f}\r".format(it,l2ErrorArray[it]))
            sys.stdout.flush()

        # the error determines the amount we need
        # to move along the derivative
        # to "regularize" this further, we multiply by the 
        # learning rate alpha

        l2Delta = ....
        l1Delta = ....

        # the weight update is the dot product between
        # the pattern input and the correction amount
        # moderated by the learning rate lr
        syn1 += ....
        syn0 += ....

    print("\noutput after training is:\n",l2)
    print("\nweights of layer1 after training are:\n",syn1)
    print("\nweights of layer0 after training are:\n",syn0)
    if (plotting):
        fig,ax = plt.subplots(figsize=(8,6))
        plt.plot(l2ErrorArray)
        plt.xlabel('Iteration')
        plt.ylabel('Summed Squared Error')
        plt.grid()
    return(syn1,l2ErrorArray,l2)

In [ ]:
# these are the inputs to our neural network
# each row is one training example
X = np.array([  [0,0,1],
                [1,1,1],
                [1,0,1],
                [0,1,1] ])
    
# this is the target state we want to have, 
# our input is three numbers, our output is
# one number
# note, that each row here corresponds to the
# row in X - so we are trying to basically
# learn two class labels from data here
y = np.array([[1],
              [1],
              [0],
              [0]])

# let's call our function and do the training:
(weights,errors,_)=threeLayerBias(X,y,5,1)

Alright, now this one works. You can also clearly see the location, where the optimization kicked in to do another round.

In [ ]:
iris = load_iris()
# these are the inputs to our neural network
# each row is one training example
X = iris.data
X = (X-X.mean(axis=0))/X.std(axis=0)    
# this is the target state we want to have
y = iris.target/2

# let's call our function and do the training:
(weights,errors,pred)=threeLayerBias(X,y.reshape(-1,1),10,1)
print(weights)
fig,ax = plt.subplots(figsize=(6,4))
plt.plot(pred)
plt.plot(y)